# KLRfome synthetic methods laboratory

This notebook is a guided diagnostic for the M0–M3 methods, not a model-ranking contest. It has three modes:

- **smoke** runs one easy case to validate execution, metrics, M1 approximation diagnostics, and invariance checks.
- **core** runs 64 controlled cases to test null behavior, signal recovery, bag-size sensitivity, spatial dependence, and unequal bags.
- **targeted_v2** revisits unresolved questions with pooled out-of-fold metrics, weaker nuisance signals, greater replication, corrected spatial dependence, and a moment-matched nonlinear test.

The unit of scientific replication is an independently generated synthetic case. Cross-validation folds within a case are averaged before uncertainty is calculated. Scores represent relative ranking, not occurrence probability.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from klrfome.utils.evaluation import replicate_summary

REPOSITORY = Path.cwd()
if not (REPOSITORY / 'benchmarks').exists():
    REPOSITORY = REPOSITORY.parent

# Change only this value when advancing from the diagnostic run to the research run.
LAB_MODE = 'smoke'  # 'smoke', 'core', or 'targeted_v2'
RUN_EXPERIMENT = True

CONFIGURATIONS = {
    'smoke': REPOSITORY / 'benchmarks/synthetic_lab_smoke_config.json',
    'core': REPOSITORY / 'benchmarks/synthetic_lab_config.json',
    'targeted_v2': REPOSITORY / 'benchmarks/synthetic_lab_targeted_v2_config.json',
}
OUTPUTS = {
    'smoke': REPOSITORY / 'benchmark_data/synthetic_lab_smoke_notebook_results.json',
    'core': REPOSITORY / 'benchmark_data/synthetic_lab_results.json',
    'targeted_v2': REPOSITORY / 'benchmark_data/synthetic_lab_targeted_v2_results.json',
}
if LAB_MODE not in CONFIGURATIONS:
    raise ValueError("LAB_MODE must be 'smoke', 'core', or 'targeted_v2'")
CONFIGURATION = CONFIGURATIONS[LAB_MODE]
OUTPUT = OUTPUTS[LAB_MODE]
sns.set_theme(style='whitegrid', context='notebook')

## 1. Configure, run, and verify scope

Keep RUN_EXPERIMENT set to True to regenerate the selected output. Set it to False to inspect an existing result without rerunning models. Smoke, core, and targeted-v2 outputs use different files.

In [ ]:
def expand_case_metadata(configuration):
    """Reconstruct the runner's case order and preserve each experiment family."""
    records = []
    case_index = 0
    default_bag_size = int(configuration.get('bag_size', 30))
    default_replicates = int(configuration.get('replicates', 1))
    for block_index, block in enumerate(configuration['scenarios']):
        effects = block.get('effect_sizes', [block.get('effect_size', 0.75)])
        bag_sizes = block.get('bag_sizes', [block.get('bag_size', default_bag_size)])
        spatial_ranges = block.get('spatial_ranges', [block.get('spatial_range', 0.0)])
        replicates = int(block.get('replicates', default_replicates))
        if block['scenario'] == 'null':
            family = 'null_control'
        elif block.get('unequal_bag_sizes', False):
            family = 'unequal_bags'
        elif len(bag_sizes) > 1:
            family = 'bag_size_sensitivity'
        elif len(spatial_ranges) > 1:
            family = 'spatial_dependence'
        else:
            family = 'signal_recovery'
        for effect in effects:
            for bag_size in bag_sizes:
                for spatial_range in spatial_ranges:
                    for replicate in range(replicates):
                        records.append({
                            'case_index': case_index,
                            'scenario_block': block_index,
                            'experiment_family': family,
                            'replicate': replicate + 1,
                            'configured_scenario': block['scenario'],
                            'configured_effect_size': float(effect),
                            'configured_bag_size': int(bag_size),
                            'configured_spatial_range': float(spatial_range),
                            'configured_unequal_bags': bool(block.get('unequal_bag_sizes', False)),
                        })
                        case_index += 1
    return pd.DataFrame(records).set_index('case_index', drop=False)

configuration = json.loads(CONFIGURATION.read_text())
expected_cases = expand_case_metadata(configuration)
display(Markdown(
    f"**Selected mode: {LAB_MODE.upper()}**  \n"
    f"Configuration: {CONFIGURATION.name}  \n"
    f"Expanded cases: {len(expected_cases)}  \n"
    f"Output: {OUTPUT.relative_to(REPOSITORY)}"
))
if LAB_MODE == 'smoke':
    display(Markdown('**Interpretation boundary:** this run validates the pipeline only; it cannot rank methods.'))
else:
    display(Markdown('**Research run:** conclusions require replicate consistency and paired differences, not a single winning fold.'))

In [ ]:
if RUN_EXPERIMENT:
    subprocess.run(
        [
            sys.executable,
            str(REPOSITORY / 'benchmarks/run_synthetic_methods_lab.py'),
            '--config', str(CONFIGURATION),
            '--output', str(OUTPUT),
        ],
        cwd=REPOSITORY,
        check=True,
    )

if not OUTPUT.exists():
    raise FileNotFoundError('Run the experiment or select an existing output file.')
result = json.loads(OUTPUT.read_text())
if result['configuration'] != configuration:
    raise ValueError('The result configuration does not match the selected configuration.')
print(result['configuration_sha256'])
print(f"{len(result['cases'])} cases loaded from {OUTPUT.name}")

## 2. What the metrics answer

| Metric | Question | Interpretation caution |
|---|---|---|
| ROC AUC | Are presence bags ranked above background bags? | 0.5 is chance; it ignores score calibration. |
| PR AUC | How concentrated are presences near the top of the ranking? | Its chance baseline depends on prevalence. |
| Boyce | Does observed presence frequency rise with suitability? | It is legitimately undefined when folds or score bins are too small. |
| Top-5% lift | How enriched is the highest-scoring fraction? | It becomes coarse or saturated with small test folds. |
| Score separation | How far apart are mean class scores? | Compare within a method; score scales differ across methods. |
| Train–test AUC gap | Is the model fitting training structure that does not generalize? | Positive values indicate possible overfitting. |
| Paired delta from M0 | Does a method beat the exact reference on identical cases and folds? | Require consistency across independent synthetic cases. |

In [ ]:
metadata_by_index = expected_cases.to_dict(orient='index')
records = []
oof_records = []
for case in result['cases']:
    case_index = int(case['case_id'].split('-', 2)[1])
    metadata = metadata_by_index[case_index]
    scenario = case['scenario']
    for row in case['fold_results']:
        records.append({
            'case_id': case['case_id'],
            'case_index': case_index,
            'experiment_family': metadata['experiment_family'],
            'replicate': metadata['replicate'],
            'scenario': scenario['scenario'],
            'effect_size': float(scenario['effect_size']),
            'bag_size': int(scenario['bag_size']),
            'spatial_range': float(scenario['spatial_range']),
            'unequal_bag_sizes': bool(scenario['unequal_bag_sizes']),
            'seed': int(scenario['seed']),
            **row,
        })
    for row in case.get('out_of_fold_results', []):
        oof_records.append({
            'case_id': case['case_id'],
            'case_index': case_index,
            'experiment_family': metadata['experiment_family'],
            'replicate': metadata['replicate'],
            'scenario': scenario['scenario'],
            'effect_size': float(scenario['effect_size']),
            'bag_size': int(scenario['bag_size']),
            'spatial_range': float(scenario['spatial_range']),
            'unequal_bag_sizes': bool(scenario['unequal_bag_sizes']),
            'seed': int(scenario['seed']),
            **row,
        })
folds = pd.DataFrame(records)
oof = pd.DataFrame(oof_records)
if oof.empty:
    display(Markdown('**Legacy result:** pooled out-of-fold predictions are unavailable; fold means are used for compatibility.'))

method_order = [item.get('id', item['method']) for item in configuration['methods']]
if configuration.get('include_baselines', True):
    method_order.extend(['LR-mean', 'RF-mean'])
    if configuration.get('include_mean_std_baseline', True):
        method_order.append('LR-mean-std')
method_order = [method for method in method_order if method in set(folds['method'])]
palette = dict(zip(method_order, sns.color_palette('husl', n_colors=len(method_order))))
folds.head()

## 3. Integrity and metric-availability gates

These checks should pass before any methodological interpretation. Undefined Boyce values and coarse lift are reported as limitations rather than silently replaced.

In [ ]:
expected_folds = int(configuration['n_splits']) * int(configuration.get('n_repeats', 1))
observed_counts = folds.groupby(['case_id', 'method']).size()
finite_columns = ['auc', 'pr_auc', 'fit_seconds', 'predict_seconds']
finite_values = np.isfinite(folds[finite_columns].to_numpy(dtype=float)).all()
expected_method_set = set(method_order)
observed_method_set = set(folds['method'])

integrity_checks = pd.DataFrame([
    {
        'gate': 'Expected case count',
        'status': 'PASS' if len(result['cases']) == len(expected_cases) else 'FAIL',
        'evidence': f"{len(result['cases'])} loaded / {len(expected_cases)} configured",
    },
    {
        'gate': 'Shared method coverage',
        'status': 'PASS' if observed_method_set == expected_method_set else 'FAIL',
        'evidence': ', '.join(sorted(observed_method_set)),
    },
    {
        'gate': 'Complete fold coverage',
        'status': 'PASS' if (observed_counts == expected_folds).all() else 'FAIL',
        'evidence': f"expected {expected_folds} rows per case and method",
    },
    {
        'gate': 'Finite primary outputs',
        'status': 'PASS' if finite_values else 'FAIL',
        'evidence': ', '.join(finite_columns),
    },
])
if not oof.empty:
    expected_oof_rows = len(result['cases']) * len(method_order) * int(configuration.get('n_repeats', 1))
    integrity_checks = pd.concat([
        integrity_checks,
        pd.DataFrame([{
            'gate': 'Pooled out-of-fold coverage',
            'status': 'PASS' if len(oof) == expected_oof_rows else 'FAIL',
            'evidence': f"{len(oof)} loaded / {expected_oof_rows} expected method-repeat rows",
        }]),
    ], ignore_index=True)
display(integrity_checks)

metric_rows = oof if not oof.empty else folds
availability = pd.DataFrame({
    'metric': ['ROC AUC', 'PR AUC', 'Boyce', 'Top-5% lift'],
    'defined_fraction': [
        metric_rows['auc'].notna().mean(),
        metric_rows['pr_auc'].notna().mean(),
        metric_rows['boyce'].notna().mean(),
        metric_rows['top_5_percent_lift'].notna().mean(),
    ],
})
display(availability.style.format({'defined_fraction': '{:.1%}'}))

top_count = (
    int(oof['top_5_percent_selected'].min())
    if not oof.empty
    else max(1, int(np.ceil(0.05 * int(folds['n_test'].min()))))
)
top_count_max = int(oof['top_5_percent_selected'].max()) if not oof.empty else top_count
if metric_rows['boyce'].notna().mean() < 0.5:
    display(Markdown('**Boyce caution:** most values are undefined; prioritize AUC and PR AUC for this run.'))
if top_count == 1:
    display(Markdown('**Lift caution:** the smallest fold selects only one bag in the top 5%, so lift is highly discrete.'))

## 4. Case-level aggregation and paired comparisons

Fold scores are averaged within each generated dataset first. Confidence intervals below are then calculated across independent synthetic cases with the same condition. This avoids treating correlated folds as independent evidence.

In [ ]:
case_keys = [
    'case_id', 'case_index', 'experiment_family', 'replicate', 'scenario',
    'effect_size', 'bag_size', 'spatial_range', 'unequal_bag_sizes', 'method',
]
fold_case_diagnostics = (
    folds.groupby(case_keys, dropna=False)
    .agg(
        train_auc=('train_auc', 'mean'),
        generalization_gap=('auc_generalization_gap', 'mean'),
    )
    .reset_index()
)
if oof.empty:
    case_scores = (
        folds.groupby(case_keys, dropna=False)
        .agg(
            auc=('auc', 'mean'),
            pr_auc=('pr_auc', 'mean'),
            boyce=('boyce', 'mean'),
            lift=('top_5_percent_lift', 'mean'),
            score_separation=('score_separation', 'mean'),
        )
        .reset_index()
    )
else:
    case_scores = (
        oof.groupby(case_keys, dropna=False)
        .agg(
            auc=('auc', 'mean'),
            pr_auc=('pr_auc', 'mean'),
            boyce=('boyce', 'mean'),
            lift=('top_5_percent_lift', 'mean'),
            score_separation=('score_separation', 'mean'),
        )
        .reset_index()
    )
case_scores = case_scores.merge(
    fold_case_diagnostics,
    on=case_keys,
    how='left',
    validate='one_to_one',
)
m0_reference = case_scores.loc[case_scores['method'] == 'M0', ['case_id', 'auc']].rename(
    columns={'auc': 'm0_auc'}
)
case_scores = case_scores.merge(m0_reference, on='case_id', how='left', validate='many_to_one')
case_scores['delta_auc_vs_M0'] = case_scores['auc'] - case_scores['m0_auc']

condition_keys = [
    'experiment_family', 'scenario', 'effect_size', 'bag_size',
    'spatial_range', 'unequal_bag_sizes', 'method',
]
comparison = (
    case_scores.groupby(condition_keys, dropna=False)
    .agg(
        n_cases=('case_id', 'nunique'),
        mean_auc=('auc', 'mean'),
        sd_auc=('auc', 'std'),
        mean_delta_auc=('delta_auc_vs_M0', 'mean'),
        sd_delta_auc=('delta_auc_vs_M0', 'std'),
        mean_gap=('generalization_gap', 'mean'),
    )
    .reset_index()
)
interval_records = []
for condition, group in case_scores.groupby(condition_keys, dropna=False):
    interval = replicate_summary(group['delta_auc_vs_M0'].to_numpy())
    record = dict(zip(condition_keys, condition))
    record['replicate_values'] = interval['values']
    record['delta_se'] = interval['standard_error']
    bounds = interval['confidence_interval']
    record['delta_ci_low'] = bounds[0] if bounds is not None else np.nan
    record['delta_ci_high'] = bounds[1] if bounds is not None else np.nan
    interval_records.append(record)
comparison = comparison.merge(
    pd.DataFrame(interval_records),
    on=condition_keys,
    how='left',
    validate='one_to_one',
)

def interval_half_width(values):
    interval = replicate_summary(np.asarray(values, dtype=float))
    bounds = interval['confidence_interval']
    return 0.0 if bounds is None else float(interval['mean'] - bounds[0])

comparison.sort_values(['experiment_family', 'scenario', 'effect_size', 'method']).head(12)

## 5. Null behavior and signal recovery

The null panel asks whether methods remain near chance when no signal exists. Signal panels ask whether performance rises with effect size. The heatmap is paired against M0: positive cells favor the row's method, negative cells favor M0. A single case is descriptive; replicate-level intervals are required for evidence.

In [ ]:
signal = case_scores[case_scores['experiment_family'].isin(['null_control', 'signal_recovery'])]
scenarios = list(dict.fromkeys(signal['scenario']))
if not scenarios:
    display(Markdown('No null or signal-recovery cases were loaded.'))
else:
    ncols = min(3, len(scenarios))
    nrows = int(np.ceil(len(scenarios) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.4 * ncols, 3.5 * nrows), squeeze=False)
    for ax, scenario_name in zip(axes.flat, scenarios):
        subset = signal[signal['scenario'] == scenario_name]
        for method in method_order:
            method_data = subset[subset['method'] == method]
            if method_data.empty:
                continue
            stats = method_data.groupby('effect_size')['auc'].agg(['mean', 'count']).reset_index()
            error = (
                method_data.groupby('effect_size')['auc']
                .apply(interval_half_width)
                .reindex(stats['effect_size'])
                .to_numpy()
            )
            ax.errorbar(
                stats['effect_size'], stats['mean'], yerr=error,
                marker='o', capsize=3, label=method, color=palette[method],
            )
        ax.axhline(0.5, color='0.45', linewidth=1, linestyle='--')
        ax.set(title=scenario_name.replace('_', ' ').title(), xlabel='Effect size', ylabel='Case-mean ROC AUC')
        ax.set_ylim(0.0, 1.02)
    for ax in axes.flat[len(scenarios):]:
        ax.set_visible(False)
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=min(5, len(labels)))
    plt.tight_layout()
    plt.show()

null_scores = case_scores[case_scores['experiment_family'] == 'null_control']
if null_scores.empty:
    display(Markdown('**Null control not included in this mode.** This is expected for smoke mode.'))
else:
    null_summary = null_scores.groupby('method')['auc'].agg(['count', 'mean', 'std']).reindex(method_order)
    null_summary['distance_from_chance'] = null_summary['mean'] - 0.5
    display(null_summary.style.format('{:.3f}', na_rep='—'))

paired_signal = comparison[
    comparison['experiment_family'].isin(['null_control', 'signal_recovery'])
    & (comparison['method'] != 'M0')
].copy()
if not paired_signal.empty:
    paired_signal['condition'] = (
        paired_signal['scenario'] + ' | effect=' + paired_signal['effect_size'].map(lambda value: f'{value:g}')
    )
    heatmap = paired_signal.pivot_table(
        index='condition', columns='method', values='mean_delta_auc', aggfunc='mean'
    ).reindex(columns=[method for method in method_order if method != 'M0'])
    plt.figure(figsize=(max(8, 1.05 * len(heatmap.columns)), max(2.8, 0.48 * len(heatmap))))
    sns.heatmap(heatmap, annot=True, fmt='.2f', center=0.0, vmin=-0.5, vmax=0.5, cmap='vlag')
    plt.title('Mean paired ROC AUC difference from M0')
    plt.xlabel('Method')
    plt.ylabel('Synthetic condition')
    plt.tight_layout()
    plt.show()

## 6. Bag size, spatial dependence, and unequal bags

These experiment families are kept separate. Declining performance for small or unequal bags points toward embedding shrinkage. Declining performance with spatial range measures sensitivity to reduced effective information, not a change in the marginal distribution.

In [ ]:
nuisance_families = [
    ('bag_size_sensitivity', 'bag_size', 'Bag-size sensitivity'),
    ('spatial_dependence', 'spatial_range', 'Spatial-dependence sensitivity'),
    ('unequal_bags', None, 'Unequal-bag robustness'),
]
available_families = [item for item in nuisance_families if item[0] in set(case_scores['experiment_family'])]
if not available_families:
    display(Markdown('No nuisance-condition cases were loaded. This is expected for smoke mode.'))
else:
    fig, axes = plt.subplots(1, len(available_families), figsize=(5.0 * len(available_families), 4), squeeze=False)
    for ax, (family, x_column, title) in zip(axes.flat, available_families):
        subset = case_scores[case_scores['experiment_family'] == family]
        if x_column is None:
            sns.boxplot(data=subset, x='method', y='auc', order=method_order, ax=ax)
            sns.stripplot(data=subset, x='method', y='auc', order=method_order, color='0.25', size=3, ax=ax)
            ax.tick_params(axis='x', rotation=55)
        else:
            for method in method_order:
                method_data = subset[subset['method'] == method]
                stats = method_data.groupby(x_column)['auc'].agg(['mean', 'count']).reset_index()
                error = (
                    method_data.groupby(x_column)['auc']
                    .apply(interval_half_width)
                    .reindex(stats[x_column])
                    .to_numpy()
                )
                ax.errorbar(
                    stats[x_column], stats['mean'], yerr=error,
                    marker='o', capsize=3, label=method, color=palette[method],
                )
        ax.axhline(0.5, color='0.45', linewidth=1, linestyle='--')
        ax.set_title(title)
        ax.set_ylabel('Case-mean ROC AUC')
        ax.set_ylim(0.0, 1.02)
    if any(item[1] is not None for item in available_families):
        handles, labels = axes.flat[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.04), ncol=min(5, len(labels)))
    plt.tight_layout()
    plt.show()

## 7. M1 approximation fidelity

M1 is primarily judged as a scalable approximation to M0. Kernel error should decrease and score-rank agreement should increase with more random features. The displayed thresholds—relative kernel error at most 0.10 and score Spearman correlation at least 0.95—are practical diagnostics, not formal statistical guarantees.

In [ ]:
rff_features = {
    item.get('id', item['method']): item.get('rff_features')
    for item in configuration['methods'] if item['method'] == 'M1'
}
m1_rows = []
for case in result['cases']:
    case_index = int(case['case_id'].split('-', 2)[1])
    metadata = metadata_by_index[case_index]
    for row in case['fold_results']:
        approximation = row.get('diagnostics', {}).get('kernel_approximation')
        agreement = row.get('diagnostics', {}).get('score_agreement_with_M0')
        if approximation and agreement:
            m1_rows.append({
                'case_id': case['case_id'],
                'experiment_family': metadata['experiment_family'],
                'method': row['method'],
                'rff_features': rff_features.get(row['method']),
                **approximation,
                **{f'score_{key}': value for key, value in agreement.items()},
            })

m1_folds = pd.DataFrame(m1_rows)
if m1_folds.empty:
    display(Markdown('No M1 approximation diagnostics were recorded.'))
    m1_summary = pd.DataFrame()
else:
    m1_cases = (
        m1_folds.groupby(['case_id', 'method', 'rff_features'], dropna=False)
        .mean(numeric_only=True)
        .reset_index()
    )
    m1_summary = (
        m1_cases.groupby(['method', 'rff_features'], dropna=False)
        .agg(
            n_cases=('case_id', 'nunique'),
            relative_kernel_error=('relative_frobenius_error', 'mean'),
            maximum_kernel_error=('relative_frobenius_error', 'max'),
            kernel_correlation=('upper_triangle_correlation', 'mean'),
            score_spearman=('score_spearman', 'mean'),
            minimum_score_spearman=('score_spearman', 'min'),
            top_5_overlap=('score_top_5_percent_overlap', 'mean'),
        )
        .reset_index()
    )
    mean_fidelity = (
        (m1_summary['relative_kernel_error'] <= 0.10)
        & (m1_summary['score_spearman'] >= 0.95)
    )
    worst_case_fidelity = (
        (m1_summary['maximum_kernel_error'] <= 0.10)
        & (m1_summary['minimum_score_spearman'] >= 0.95)
    )
    m1_summary['diagnostic_status'] = np.select(
        [mean_fidelity & worst_case_fidelity, mean_fidelity],
        ['HIGH FIDELITY', 'PRACTICAL; REVIEW TAILS'],
        default='REVIEW',
    )
    display(m1_summary.style.format({
        'relative_kernel_error': '{:.3f}',
        'maximum_kernel_error': '{:.3f}',
        'kernel_correlation': '{:.3f}',
        'score_spearman': '{:.3f}',
        'minimum_score_spearman': '{:.3f}',
        'top_5_overlap': '{:.1%}',
    }))

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    sns.lineplot(data=m1_cases, x='rff_features', y='relative_frobenius_error', marker='o', ax=axes[0])
    axes[0].axhline(0.10, color='0.45', linestyle='--', linewidth=1)
    axes[0].set(title='Kernel approximation error', ylabel='Relative Frobenius error')
    sns.lineplot(data=m1_cases, x='rff_features', y='score_spearman', marker='o', ax=axes[1])
    axes[1].axhline(0.95, color='0.45', linestyle='--', linewidth=1)
    axes[1].set(title='Prediction-rank agreement with M0', ylabel='Spearman correlation', ylim=(0, 1.02))
    for ax in axes:
        ax.set_xscale('log', base=2)
    plt.tight_layout()
    plt.show()

## 8. Distribution invariance checks

Cell order and uniform duplication should not change predictions beyond numerical precision. Selectively duplicating cells changes empirical probability mass and is therefore expected to change predictions. It is reported as sensitivity, not scored as pass or fail.

In [ ]:
invariance_rows = []
for case in result['cases']:
    for method, values in (case.get('invariance') or {}).items():
        invariance_rows.append({
            'case_id': case['case_id'],
            'method': method,
            'permutation_change': values['permuted_maximum_absolute_score_change'],
            'uniform_duplication_change': values['uniformly_duplicated_maximum_absolute_score_change'],
            'selective_duplication_change': values['selectively_duplicated_maximum_absolute_score_change'],
        })
invariance = pd.DataFrame(invariance_rows)
if invariance.empty:
    display(Markdown('Invariance checks were disabled in this configuration. The smoke run is the required invariance gate.'))
    invariance_pass = None
else:
    invariance_summary = invariance.groupby('method').agg(
        maximum_permutation_change=('permutation_change', 'max'),
        maximum_uniform_duplication_change=('uniform_duplication_change', 'max'),
        median_selective_duplication_change=('selective_duplication_change', 'median'),
    ).reindex([method for method in method_order if method in set(invariance['method'])])
    invariance_summary['invariance_status'] = np.where(
        (invariance_summary['maximum_permutation_change'] <= 1e-5)
        & (invariance_summary['maximum_uniform_duplication_change'] <= 1e-5),
        'PASS',
        'FAIL',
    )
    invariance_pass = bool((invariance_summary['invariance_status'] == 'PASS').all())
    display(invariance_summary.style.format({
        'maximum_permutation_change': '{:.2e}',
        'maximum_uniform_duplication_change': '{:.2e}',
        'median_selective_duplication_change': '{:.3f}',
    }))

## 9. Computational diagnostics

Timing is descriptive. The first fold can include JAX tracing and compilation, while new bag shapes may trigger later compilations. Use later-fold medians as a better—but still approximate—steady-state comparison. Dedicated scaling benchmarks remain necessary for speed claims.

In [ ]:
first_fold_mask = (folds['repeat'] == 1) & (folds['fold'] == 1)
timing_all = folds.groupby('method').agg(
    median_fit_seconds=('fit_seconds', 'median'),
    median_predict_seconds=('predict_seconds', 'median'),
    median_peak_python_mb=('peak_python_memory_mb', 'median'),
)
first_timing = folds[first_fold_mask].groupby('method')['fit_seconds'].median().rename('first_fold_fit_seconds')
later_timing = folds[~first_fold_mask].groupby('method')['fit_seconds'].median().rename('later_fold_fit_seconds')
timing = timing_all.join(first_timing).join(later_timing).reindex(method_order)
display(timing.style.format('{:.4f}', na_rep='—'))

## 10. Automated interpretation and extension gate

The table below identifies evidence and limitations; it deliberately avoids declaring a universal winner. A method earns further consideration when it improves on the same cases as M0, generalizes, and repeats across independently generated datasets.

In [ ]:
findings = []
integrity_pass = bool((integrity_checks['status'] == 'PASS').all())
findings.append({
    'gate': 'Execution integrity',
    'status': 'PASS' if integrity_pass else 'FAIL',
    'evidence': f"{len(result['cases'])} cases; {len(folds)} fold-method rows",
    'meaning': 'Interpret results only if all integrity checks pass.',
})
findings.append({
    'gate': 'Scientific scope',
    'status': 'DIAGNOSTIC ONLY' if LAB_MODE == 'smoke' else 'RESEARCH SUITE',
    'evidence': f"{len(result['cases'])} independently generated cases",
    'meaning': 'Smoke validates software; core records broad behavior; targeted v2 resolves identified limitations.',
})
boyce_fraction = float(metric_rows['boyce'].notna().mean())
findings.append({
    'gate': 'Presence-only metrics',
    'status': 'REVIEW' if boyce_fraction < 0.5 or top_count == 1 else 'USABLE',
    'evidence': f"Boyce defined {boyce_fraction:.0%}; top 5% selects {top_count}–{top_count_max} bag(s)",
    'meaning': 'Prefer AUC and PR AUC when Boyce is sparse or lift is discrete.',
})
if not m1_summary.empty:
    for row in m1_summary.itertuples():
        findings.append({
            'gate': f'M1 fidelity: {row.method}',
            'status': row.diagnostic_status,
            'evidence': f"features={int(row.rff_features)}, mean/max error={row.relative_kernel_error:.3f}/{row.maximum_kernel_error:.3f}, mean/min rank={row.score_spearman:.3f}/{row.minimum_score_spearman:.3f}",
            'meaning': 'Review each feature budget separately; one low-budget variant must not fail the whole M1 family.',
        })
if invariance_pass is not None:
    findings.append({
        'gate': 'Distribution invariance',
        'status': 'PASS' if invariance_pass else 'FAIL',
        'evidence': 'Permutation and uniform-duplication tolerance = 1e-5',
        'meaning': 'Selective duplication is intentionally excluded because it changes probability mass.',
    })
if not null_scores.empty:
    null_deviation = null_scores.groupby('method')['auc'].mean().sub(0.5).abs()
    findings.append({
        'gate': 'Null behavior',
        'status': 'CONSISTENT' if null_deviation.max() <= 0.15 else 'REVIEW',
        'evidence': f"largest mean distance from chance = {null_deviation.max():.3f}",
        'meaning': 'Systematic null performance away from 0.5 suggests leakage, bias, or too few replicates.',
    })

robust_gains = comparison[
    (comparison['method'] != 'M0')
    & (comparison['n_cases'] >= 2)
    & (comparison['mean_delta_auc'] > 0.02)
    & (comparison['delta_ci_low'] > 0)
]
if LAB_MODE != 'smoke':
    if robust_gains.empty:
        gain_evidence = 'No paired gain above 0.02 has a replicate-level interval entirely above zero.'
        gain_status = 'NO CLEAR GAIN'
    else:
        gain_evidence = '; '.join(
            f"{row.scenario}/{row.method}: ΔAUC={row.mean_delta_auc:.3f}"
            for row in robust_gains.sort_values('mean_delta_auc', ascending=False).head(6).itertuples()
        )
        gain_status = 'CANDIDATES'
    findings.append({
        'gate': 'Replicated gains over M0',
        'status': gain_status,
        'evidence': gain_evidence,
        'meaning': 'Candidates still require scenario coherence and acceptable generalization gaps.',
    })
display(pd.DataFrame(findings))

extension_guide = pd.DataFrame([
    {'Observed pattern': 'M1 kernel error remains high while M0 succeeds', 'Next extension': 'Improved random features'},
    {'Observed pattern': 'Performance collapses for small or unequal bags', 'Next extension': 'Shrinkage embeddings'},
    {'Observed pattern': 'Sparse-signal cases trail comparable low-dimensional cases', 'Next extension': 'ARD or grouped kernels'},
    {'Observed pattern': 'M2 repeatedly improves nonlinear-mixture cases without a large gap', 'Next extension': 'Continue nonlinear bag-level kernels'},
    {'Observed pattern': 'M3 repeatedly improves variance, tail, multimodal, or correlation cases', 'Next extension': 'Continue Wasserstein or hybrid kernels'},
])
display(Markdown('### How findings select the next methods sprint'))
display(extension_guide)

## 11. Advancement checklist

After the smoke run, advance only if execution integrity passes, M1 diagnostics are finite, and invariance passes. Boyce may remain undefined and lift may remain saturated in smoke mode; those are expected consequences of the tiny test folds.

To run the historical 64-case suite, change LAB_MODE from smoke to core. To answer the unresolved small-bag, spatial-dependence, sparse-signal, and moment-matched nonlinear questions, use targeted_v2. Keep RUN_EXPERIMENT set to True for the first run of a mode. After it completes, set RUN_EXPERIMENT to False while revisiting plots so the expensive models are not rerun.

Do not promote or remove a method based on synthetic performance alone. Use the laboratory to determine whether a method behaves coherently under signals it claims to represent, then return to mapped empirical validation.